# CerviGrade: detector plus CNN on whole CRIC slides
**The idea.** The detector slides a window over a whole slide and boxes every cell. Each box is cut out the way the CNNs were trained (a 200 x 200 crop around the cell), and the CNN grades it. The slide's verdict is the worst grade the CNN is sure about.

**Why.** The detector alone finds only about 4 in 10 marked cells and grades only about 3 in 10 correctly. The CNNs grade about 8 in 10 pre-cropped cells. This tests whether the CNN fixes the grading half. It cannot fix the finding half, so about 40% found is the ceiling.

**One fair test.** Only validation and test slides are used, and they are the same slides the CNNs never trained on. The two cut-offs (detector confidence, CNN confidence) are picked on the validation slides. The test slides are scored once with them.

**Inputs to add:** the CRIC dataset that holds the full slides and `classifications.csv`, the Kaggle Dataset with the detector `best.pt` files, and the Dataset with the retrain results (the three CNN files, `retrain_selection.json` and `runs.json`). GPU on, Internet on for the pip line. Run `pip install ultralytics` first if it is missing.

In [ ]:
!pip install -q ultralytics

## 1. Settings
The only cell you edit. For a quick check, set `MAX_SLIDES = 6`.

In [ ]:
# ----- Settings (the only cell you edit) -----
DETECTOR_MATCH = ""        # part of a weights path to pick one detector, such as "oversampled". Empty means every 6-class detector found.
CNN_NAMES = ["EfficientNetB0", "ConvNeXt", "RegNet"]   # a model whose file is not found is skipped
DET_LOW_CONF = 0.25        # the detector keeps every box above this. The cut-offs below are tried afterwards.
DET_CONFS = [0.25, 0.4, 0.5, 0.6]   # detector confidence cut-offs to try
CNN_CONFS = [0.5, 0.7, 0.9]         # how sure the CNN must be before a box counts as abnormal
TILE, STRIDE, CROP = 224, 112, 200  # detector window, its step, and the size of the cell crops the CNNs trained on
MAX_SLIDES = None          # a number to use only the first few slides of each split, for a quick check

## 1b. Recap of the retrain results
The tables from the retrain run, read from your Dataset. This is the performance of the three CNNs on pre-cropped cells. Nothing here is recomputed.

In [ ]:
# ----- Recap of the retrain results (read from the Dataset, nothing is recomputed) -----
import glob, json, os
import pandas as pd
from IPython.display import display

recap_hits = glob.glob("/kaggle/input/**/retrain_summary.csv", recursive=True)
if not recap_hits:
    print("retrain_summary.csv not found. Attach the retrain results Dataset to see the recap.")
else:
    recap_dir = os.path.dirname(recap_hits[0])
    print("Retrain results read from", recap_dir, "(CRIC test, 1,723 cells, 3 seeds per model)")
    print("\nAverage ± standard deviation across seeds")
    display(pd.read_csv(recap_hits[0], index_col=0))
    print("Every run")
    display(pd.read_csv(os.path.join(recap_dir, "retrain_all_runs.csv")))
    recap_sel = os.path.join(recap_dir, "retrain_selection.json")
    if os.path.exists(recap_sel):
        print("Seed kept for each model (best validation macro F1):", json.load(open(recap_sel))["best_seed"])

## 2. Find the slides and rebuild the split
Same split as the training notebooks: by source slide, seed 42, 70 / 15 / 15. On the full data the test split has 60 slides and 1,723 cells.

In [ ]:
import os, glob, json, random, time, pickle
import numpy as np
import pandas as pd

INPUT_ROOT = "/kaggle/input"

def short(name):
    return "Negative" if name.startswith("Negative") else name

def figshare_id_from_doi(doi):
    return doi.rstrip("/").split("figshare.")[-1]

csv_path = next((os.path.join(d, "classifications.csv") for d, _, f in os.walk(INPUT_ROOT) if "classifications.csv" in f), None)
assert csv_path, "classifications.csv not found: attach the CRIC dataset that holds the full slides."
CRIC_ROOT = os.path.dirname(csv_path)
CRIC_IMAGES_DIR = os.path.join(CRIC_ROOT, "images")
assert os.path.isdir(CRIC_IMAGES_DIR), f"No images folder at {CRIC_IMAGES_DIR}"

cric = pd.read_csv(csv_path)
cric["on_disk"] = cric["image_doi"].apply(figshare_id_from_doi) + "_" + cric["image_filename"]
cric["group"] = cric["image_id"].astype(str)

def split_by_group_global(groups_all, seed=42, train_frac=0.7, val_frac=0.15):
    # Same logic and seed as the training notebooks, so every split holds the same slides.
    rng = random.Random(seed)
    groups = sorted(set(groups_all))
    rng.shuffle(groups)
    n = len(groups)
    n_train, n_val = round(n * train_frac), round(n * val_frac)
    out = {}
    for g in groups[:n_train]: out[g] = "train"
    for g in groups[n_train:n_train + n_val]: out[g] = "val"
    for g in groups[n_train + n_val:]: out[g] = "test"
    return out

cric["split"] = cric["group"].map(split_by_group_global(cric["group"]))
CLASSES = sorted(cric["bethesda_system"].unique())          # same order as the CNNs' outputs
SHORT_NAMES = [short(c) for c in CLASSES]
assert len(CLASSES) == 6, CLASSES

slides = {s: cric[cric["split"] == s]["on_disk"].drop_duplicates().tolist() for s in ("val", "test")}
if MAX_SLIDES:
    slides = {s: v[:MAX_SLIDES] for s, v in slides.items()}
cells = {s: cric[(cric["split"] == s) & cric["on_disk"].isin(slides[s])] for s in ("val", "test")}
cells_by_slide = {s: g for split in cells for s, g in cells[split].groupby("on_disk")}
for s in ("val", "test"):
    print(f"{s}: {len(slides[s])} slides, {len(cells[s])} expert-marked cells")
print("On the full data the test split has 60 slides and 1,723 cells, and val has 1,807 cells.")

## 3. Helpers
The window scan is the same one used in the earlier slide test. `crop_at` cuts the same 200 x 200 window the CNNs trained on.

In [ ]:
import cv2
import torch
from torchvision.ops import nms

RANK = {"Negative": 0, "ASC-US": 1, "LSIL": 2, "ASC-H": 3, "HSIL": 4, "SCC": 5}
KIND_NAMES = ["healthy", "mild", "serious"]        # healthy = Negative, mild = ASC-US or LSIL, serious = ASC-H, HSIL or SCC
CLASS_RANK = np.array([RANK[n] for n in SHORT_NAMES])

def kind(rank):
    return 0 if rank == 0 else (1 if rank <= 2 else 2)

def tile_starts(length, tile, stride):
    last = max(length - tile, 0)
    starts = list(range(0, last + 1, stride))
    if starts[-1] != last:
        starts.append(last)
    return starts

def point_in_box(px, py, box):
    x1, y1, x2, y2 = box
    return x1 <= px <= x2 and y1 <= py <= y2

def load_slide(on_disk):
    return cv2.imread(os.path.join(CRIC_IMAGES_DIR, on_disk))

def detect_field(model, img_bgr, conf=DET_LOW_CONF):
    """Slide a window over the photo, let the detector box every cell, merge boxes found twice."""
    H, W = img_bgr.shape[:2]
    tiles, offsets = [], []
    for y in tile_starts(H, TILE, STRIDE):
        for x in tile_starts(W, TILE, STRIDE):
            tiles.append(img_bgr[y:y + TILE, x:x + TILE])
            offsets.append((x, y))
    results = model.predict(tiles, imgsz=224, conf=conf, verbose=False, batch=64)
    boxes, scores, classes = [], [], []
    for r, (ox, oy) in zip(results, offsets):
        if r.boxes is None or len(r.boxes) == 0:
            continue
        xyxy = r.boxes.xyxy.cpu().numpy().copy()
        xyxy[:, [0, 2]] += ox
        xyxy[:, [1, 3]] += oy
        boxes.append(xyxy)
        scores.append(r.boxes.conf.cpu().numpy())
        classes.append(r.boxes.cls.cpu().numpy().astype(int))
    if not boxes:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, dtype=int)
    boxes = np.concatenate(boxes); scores = np.concatenate(scores); classes = np.concatenate(classes)
    keep = nms(torch.tensor(boxes), torch.tensor(scores), 0.5).numpy()      # one box per cell, whatever its grade
    return boxes[keep], scores[keep], classes[keep]

def crop_at(img_bgr, cx, cy):
    """The same 200 x 200 window the CNNs were trained on: centred on the point, pushed back inside the photo at the edges."""
    h, w = img_bgr.shape[:2]
    left = max(0, min(int(cx) - CROP // 2, w - CROP))
    top = max(0, min(int(cy) - CROP // 2, h - CROP))
    return img_bgr[top:top + CROP, left:left + CROP]

## 4. Load the detector and the retrained CNNs
The detector is only used to find cells. The CNN files come from the retrain notebook's output, one saved seed per model (the one with the best validation score). A model whose file is missing is skipped, so RegNet can be added later.

In [ ]:
import tensorflow as tf
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.set_logical_device_configuration(gpu, [tf.config.LogicalDeviceConfiguration(memory_limit=8192)])
except Exception as e:
    print("GPU memory limit left as it was:", e)
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_pre
from tensorflow.keras.applications.convnext import preprocess_input as convnext_pre
from ultralytics import YOLO
from torchvision.models import regnet_y_400mf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", device)

# --- The detector: it only has to find cells here, so its own grade is ignored except in the "detector alone" row ---
detectors = {}
for p in sorted(glob.glob(f"{INPUT_ROOT}/**/*.pt", recursive=True)):
    if os.path.basename(p).lower().startswith("regnet") or (DETECTOR_MATCH and DETECTOR_MATCH not in p):
        continue
    try:
        m = YOLO(p)
    except Exception:
        continue
    if len(m.names) == 6:
        detectors[p] = m
        print("DETECTOR:", p, [short(n) for n in m.names.values()])
assert detectors, "No 6-class detector weights found. Attach the Kaggle Dataset that holds your best.pt files."
det_rank = {p: np.array([RANK[short(m.names[i])] for i in range(6)]) for p, m in detectors.items()}
def det_label(p):
    parts = p.split("/")
    return parts[-3] if len(parts) >= 3 and parts[-1].endswith(".pt") and parts[-2] == "weights" else os.path.basename(p)

# --- The graders: the retrained CNNs from the retrain notebook's output ---
CNN_FILES = {"EfficientNetB0": "efficientnetb0_cric_retrained.keras",
             "ConvNeXt": "convnext_cric_retrained.keras",
             "RegNet": "regnet_cric_retrained.pt"}
KERAS_PRE = {"EfficientNetB0": effnet_pre, "ConvNeXt": convnext_pre}
cnns = {}
for name in CNN_NAMES:
    fname = CNN_FILES[name]
    path = next((os.path.join(d, fname) for d, _, f in os.walk(INPUT_ROOT) if fname in f), None)
    print(f"{name}: {'found' if path else 'NOT FOUND, skipped'}")
    if not path:
        continue
    if name == "RegNet":
        m = regnet_y_400mf(weights=None)
        m.fc = torch.nn.Sequential(torch.nn.Dropout(0.3), torch.nn.Linear(m.fc.in_features, 6))
        m.load_state_dict(torch.load(path, map_location="cpu"))
        cnns[name] = m.eval().to(device)
    else:
        cnns[name] = tf.keras.models.load_model(path)
assert cnns, "No retrained CNN files found. Attach the Dataset with the retrain results (Add Input)."

# test accuracy each saved model scored in the retrain notebook, to check against below
ref_acc = {}
sel = next(iter(glob.glob(f"{INPUT_ROOT}/**/retrain_selection.json", recursive=True)), None)
runs_file = next(iter(glob.glob(os.path.join(os.path.dirname(sel), "runs.json")) + glob.glob(os.path.join(os.path.dirname(sel), "runs_*", "runs.json"))), None) if sel else None
if sel and runs_file:
    best_seed, runs = json.load(open(sel))["best_seed"], json.load(open(runs_file))
    ref_acc = {a: runs[f"{a}_seed{s}"]["accuracy"] for a, s in best_seed.items() if f"{a}_seed{s}" in runs}

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def grade(name, crops_bgr, batch=128):
    """Class probabilities, shape (n, 6), for a list of BGR crops."""
    if not len(crops_bgr):
        return np.zeros((0, 6), dtype=np.float32)
    out = []
    for i in range(0, len(crops_bgr), batch):
        x = np.stack([cv2.resize(cv2.cvtColor(c, cv2.COLOR_BGR2RGB).astype(np.float32), (224, 224), interpolation=cv2.INTER_LINEAR)
                      for c in crops_bgr[i:i + batch]])
        if name == "RegNet":
            t = torch.from_numpy(((x / 255.0 - MEAN) / STD).transpose(0, 3, 1, 2).copy()).to(device)
            with torch.no_grad():
                out.append(torch.softmax(cnns[name](t), dim=1).cpu().numpy())
        else:
            out.append(cnns[name].predict(KERAS_PRE[name](x), batch_size=batch, verbose=0))
    return np.concatenate(out)

## 5. Check the cropping
Grades crops centred on the experts' dots. The accuracy should match what the retrain notebook reported for each model. If it does not, stop and tell me, because the numbers below would not be trustworthy.

In [ ]:
# Grade crops centred on the experts' dots, the way the CNNs were tested in the retrain notebook.
# If this accuracy matches, the cropping and preprocessing here are right. It is also the best the grader can do with perfect boxes.
ideal = {}
crops_ideal, y_ideal = [], []
for s in slides["test"]:
    img = load_slide(s)
    for _, r in cells_by_slide[s].iterrows():
        crops_ideal.append(crop_at(img, r["nucleus_x"], r["nucleus_y"]))
        y_ideal.append(CLASSES.index(r["bethesda_system"]))
y_ideal = np.array(y_ideal)
for name in cnns:
    pred = grade(name, crops_ideal).argmax(1)
    scc = SHORT_NAMES.index("SCC")
    ideal[name] = {"right": int((pred == y_ideal).sum()), "total": len(y_ideal),
                   "scc_right": int(((pred == y_ideal) & (y_ideal == scc)).sum()), "scc_total": int((y_ideal == scc).sum())}
    acc = ideal[name]["right"] / ideal[name]["total"]
    note = ""
    if name in ref_acc:
        note = f" | retrain notebook said {ref_acc[name]:.3f}" + ("" if abs(acc - ref_acc[name]) < 0.03 else "   <-- MISMATCH, check the crops")
    print(f"{name}: accuracy on the {len(y_ideal)} expert-marked test cells = {acc:.3f}{note}")

## 6. Let the detector scan every validation and test slide
Keeps every box above the lowest confidence. Stricter cut-offs are applied afterwards without scanning again.

In [ ]:
from tqdm.auto import tqdm
dets = {}      # (detector, slide) -> boxes, scores, classes
for dpath, model in detectors.items():
    t0 = time.time()
    for split in ("val", "test"):
        for s in tqdm(slides[split], desc=f"{det_label(dpath)} on {split} slides", unit="slide"):
            dets[(dpath, s)] = detect_field(model, load_slide(s))
    n_boxes = sum(len(dets[(dpath, s)][0]) for split in slides for s in slides[split])
    print(f"{det_label(dpath)}: {time.time() - t0:.0f} s, {n_boxes} boxes at confidence >= {DET_LOW_CONF}")

## 7. Grade every box with every CNN

In [ ]:
probs = {}     # (detector, cnn, slide) -> (n_boxes, 6) class probabilities
for dpath in detectors:
    for split in ("val", "test"):
        for s in tqdm(slides[split], desc=f"CNNs on {det_label(dpath)} boxes ({split})", unit="slide"):
            img = load_slide(s)
            boxes = dets[(dpath, s)][0]
            crops = [crop_at(img, (b[0] + b[2]) / 2, (b[1] + b[3]) / 2) for b in boxes]
            for name in cnns:
                probs[(dpath, name, s)] = grade(name, crops)
print("Graded every box with every CNN.")

## 8. Slide level: which pipeline gets the slide's verdict right
Three slide types, as in the earlier test: healthy (only Negative cells marked), mild (worst grade ASC-US or LSIL) and serious (worst grade ASC-H, HSIL or SCC). Balanced accuracy is the average of the three recalls, so the many mild slides do not drown out the serious ones. A slide with no box counts as healthy.

In [ ]:
expert_rank = {s: max(RANK[short(c)] for c in cells_by_slide[s]["bethesda_system"])
               for split in slides for s in slides[split]}

def verdict_detector(dpath, s, det_conf):
    """The detector alone: the worst grade among its boxes."""
    boxes, scores, classes = dets[(dpath, s)]
    keep = scores >= det_conf
    return int(det_rank[dpath][classes[keep]].max()) if keep.any() else 0

def verdict_cnn(dpath, name, s, det_conf, cnn_conf):
    """Detector finds the cells, the CNN grades each one, and the worst grade it is sure about is the slide's verdict."""
    boxes, scores, classes = dets[(dpath, s)]
    keep = scores >= det_conf
    if not keep.any():
        return 0
    p = probs[(dpath, name, s)][keep]
    ranks = CLASS_RANK[p.argmax(1)]
    sure = (ranks == 0) | (p.max(1) >= cnn_conf)
    return int(ranks[sure].max()) if sure.any() else 0

def summarise(pred, split):
    s_list = slides[split]
    true_k = np.array([kind(expert_rank[s]) for s in s_list])
    pred_k = np.array([kind(pred[s]) for s in s_list])
    hits = [int((pred_k[true_k == k] == k).sum()) for k in range(3)]
    sizes = [int((true_k == k).sum()) for k in range(3)]
    recalls = [h / n for h, n in zip(hits, sizes) if n]
    return {"hits": hits, "sizes": sizes, "balanced": float(np.mean(recalls)),
            "exact": float(np.mean([pred[s] == expert_rank[s] for s in s_list])), "true_k": true_k, "pred_k": pred_k}

# Pick the settings on the VALIDATION slides, then score the TEST slides once with them.
pipelines = [(d, None) for d in detectors] + [(d, n) for d in detectors for n in cnns]
chosen = {}
for dpath, name in pipelines:
    best = None
    for dc in DET_CONFS:
        for cc in ([None] if name is None else CNN_CONFS):
            pred = {s: (verdict_detector(dpath, s, dc) if name is None else verdict_cnn(dpath, name, s, dc, cc)) for s in slides["val"]}
            m = summarise(pred, "val")
            key = (m["balanced"], m["hits"][2] / max(m["sizes"][2], 1))     # ties go to catching more serious slides
            if best is None or key > best[0]:
                best = (key, dc, cc)
    chosen[(dpath, name)] = {"det_conf": best[1], "cnn_conf": best[2]}

rows, tables = [], {}
for dpath, name in pipelines:
    c = chosen[(dpath, name)]
    pred = {s: (verdict_detector(dpath, s, c["det_conf"]) if name is None else verdict_cnn(dpath, name, s, c["det_conf"], c["cnn_conf"]))
            for s in slides["test"]}
    m = summarise(pred, "test")
    label = ("Detector alone" if name is None else f"Detector + {name}") + (f" ({det_label(dpath)})" if len(detectors) > 1 else "")
    rows.append({"pipeline": label,
                 "settings (picked on val)": f"detector >= {c['det_conf']}" + ("" if name is None else f", CNN >= {c['cnn_conf']}"),
                 "healthy kept healthy": f"{m['hits'][0]}/{m['sizes'][0]}",
                 "mild called mild": f"{m['hits'][1]}/{m['sizes'][1]}",
                 "serious flagged serious": f"{m['hits'][2]}/{m['sizes'][2]}",
                 "balanced accuracy": round(m["balanced"], 3),
                 "exact worst grade": f"{m['exact']:.0%}"})
    tables[label] = pd.crosstab(pd.Series([KIND_NAMES[k] for k in m["true_k"]], name="expert slide type"),
                                pd.Series([KIND_NAMES[k] for k in m["pred_k"]], name="pipeline says"))
from IPython.display import display
print(f"Slide level, {len(slides['test'])} test slides (settings chosen on the {len(slides['val'])} validation slides)")
slide_table = pd.DataFrame(rows)
display(slide_table)
for label, t in tables.items():
    print(label)
    display(t)

## 9. Cell level
For each expert dot: was it inside a box, and did the pipeline give it the right grade. The last rows show each CNN given the expert's dot, which is the best the grader can do.

In [ ]:
def cell_level(dpath, name, det_conf, split="test"):
    model = detectors[dpath]
    found = right = total = scc_right = scc_total = 0
    for s in slides[split]:
        boxes, scores, classes = dets[(dpath, s)]
        keep = np.where(scores >= det_conf)[0]
        for _, r in cells_by_slide[s].iterrows():
            truth = short(r["bethesda_system"])
            total += 1
            scc_total += truth == "SCC"
            hits = [i for i in keep if point_in_box(r["nucleus_x"], r["nucleus_y"], boxes[i])]
            if not hits:
                continue
            found += 1
            b = min(hits, key=lambda i: (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1]))      # the smallest box around the dot
            said = short(model.names[classes[b]]) if name is None else SHORT_NAMES[int(probs[(dpath, name, s)][b].argmax())]
            right += said == truth
            scc_right += (said == truth) and truth == "SCC"
    return found, right, total, scc_right, scc_total

rows = []
for dpath, name in pipelines:
    c = chosen[(dpath, name)]
    f, r, t, sr, st = cell_level(dpath, name, c["det_conf"])
    rows.append({"pipeline": ("Detector alone" if name is None else f"Detector + {name}") + (f" ({det_label(dpath)})" if len(detectors) > 1 else ""),
                 "detector cut-off": c["det_conf"],
                 "cells found": f"{f}/{t} ({f / t:.0%})",
                 "found and graded right": f"{r}/{max(f, 1)} ({r / max(f, 1):.0%})",
                 "graded right, of all cells": f"{r}/{t} ({r / t:.0%})",
                 "SCC graded right": f"{sr}/{st}"})
for name, v in ideal.items():
    rows.append({"pipeline": f"{name} given the expert's dot (best case)", "detector cut-off": "-",
                 "cells found": f"{v['total']}/{v['total']}", "found and graded right": f"{v['right']}/{v['total']} ({v['right'] / v['total']:.0%})",
                 "graded right, of all cells": f"{v['right']}/{v['total']} ({v['right'] / v['total']:.0%})", "SCC graded right": f"{v['scc_right']}/{v['scc_total']}"})
print(f"Cell level, {len(slides['test'])} test slides")
cell_table = pd.DataFrame(rows)
display(cell_table)

## 10. Look at one slide
Boxes are coloured by the CNN's grade, with its confidence. Yellow dots are the experts' marks. A box with no dot is not always wrong, because the experts did not mark every cell.

In [ ]:
import matplotlib.pyplot as plt
dpath0, name0 = next(((d, n) for d, n in pipelines if n is not None), (None, None))
colors = {"Negative": (0, 200, 0), "ASC-US": (255, 200, 0), "LSIL": (255, 140, 0), "ASC-H": (255, 80, 80), "HSIL": (220, 0, 0), "SCC": (160, 0, 160)}
c0 = chosen[(dpath0, name0)]
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, s in zip(axes, slides["test"][:2]):
    img = load_slide(s)
    vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    boxes, scores, classes = dets[(dpath0, s)]
    p = probs[(dpath0, name0, s)]
    for (x1, y1, x2, y2), sc, row in zip(boxes, scores, p):
        if sc < c0["det_conf"]:
            continue
        g = SHORT_NAMES[int(row.argmax())]
        col = colors[g]
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
        cv2.putText(vis, f"{g} {row.max():.2f}", (int(x1), max(int(y1) - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
    for _, r in cells_by_slide[s].iterrows():
        cv2.circle(vis, (int(r["nucleus_x"]), int(r["nucleus_y"])), 6, (255, 255, 0), -1)
    ax.imshow(vis)
    ax.set_title(f"{name0}: grade and confidence per box. Yellow dots are the experts' marks.")
    ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/combo_picture_check.png", dpi=100)
plt.show()

## 11. Save the results

In [ ]:
out = {"settings": {"det_confs": DET_CONFS, "cnn_confs": CNN_CONFS, "tile": TILE, "stride": STRIDE, "crop": CROP},
       "chosen_on_val": {f"{det_label(d)} | {n or 'detector alone'}": v for (d, n), v in chosen.items()},
       "slide_level_test": slide_table.to_dict(orient="records"),
       "cell_level_test": cell_table.to_dict(orient="records")}
json.dump(out, open("/kaggle/working/combo_results.json", "w"), indent=2, default=str)
slide_table.to_csv("/kaggle/working/combo_slide_level.csv", index=False)
cell_table.to_csv("/kaggle/working/combo_cell_level.csv", index=False)
print("Saved combo_results.json, combo_slide_level.csv and combo_cell_level.csv in /kaggle/working.")

## How to read this
- **Detector alone against Detector + CNN** is the main comparison. If the CNN rows are not clearly better on serious slides, the combination adds little.
- **False alarms are the risk.** The CNN has no "not a cell" class, so every wrong box still gets graded. If healthy slides get flagged serious, raise the CNN confidence or the detector cut-off.
- **The three CNN rows** show which grader works best in the pipeline. Small differences are noise, because there are only 60 test slides.
- **Settings were picked on validation slides**, never on the test slides.
- The detector's own limit does not change: cells it never boxes are never graded.